In [ ]:

from scipy import sparse as sp
import pandas as pd
import numpy as np
import torch

import torch.nn as nn

#保存all_sequence_outputsnew
#np.save('./data/all_sequence_outputsnew7132.npy', all_sequence_outputsnew)
all_sequence_outputsnew=np.load('./data/all_sequence_outputsnew7132.npy')
all_sequence_outputsnew.shape
ppi_matrix=pd.read_csv('./data/9606ppi_matrix.csv')
ppi_matrix = sp.coo_matrix(ppi_matrix)
# featureDF.to_csv("./data/24077132kdncmergedf.csv")
merged_df=pd.read_csv('./data/24077132kdncmergedf.csv')
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv  # 使用SAGEConv替代GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from src.utils import set_seed
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops

# 设置随机种子
SEED = 12
set_seed(SEED)
# 假定 featureDF, ppi_matrix, 和 all_sequence_outputsnew 已经定义好

# 数据标准化
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rNC2']].values / np.median(merged_df[['rNC2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['NC3']].values / np.median(merged_df[['NC3']].values))+ 1)

# 数据准备
y = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 定义 CPM 转换函数（除以中位数再乘以 1e6）
X_cpm_log2 = np.log2((merged_df[['rKD2']].values / np.median(merged_df[['rKD2']].values)) + 1)
y_cpm_log2 = np.log2((merged_df[['KD3']].values / np.median(merged_df[['KD3']].values))+ 1)

# 数据准备
y1 = torch.tensor(y_cpm_log2, dtype=torch.float32).view(-1, 1)
X1 = torch.tensor(X_cpm_log2, dtype=torch.float32).view(-1, 1)
# 转换为PyTorch Tensor
edge_index, edge_weight = from_scipy_sparse_matrix(ppi_matrix.astype('float32'))

# 创建PyTorch Geometric的Data对象
train_data = Data(
    x=X,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=y,
    seq=torch.tensor(all_sequence_outputsnew, dtype=torch.float32),
    pause=torch.tensor(merged_df['High_Pause_Countsnc'].values, dtype=torch.float32)
)
print('添加自环前：', train_data)

# 添加自环
train_data.edge_index, train_data.edge_attr = add_self_loops(train_data.edge_index, train_data.edge_weight)
print('\n添加自环后：', train_data)

# 创建测试集Data对象
test_data = Data(
    x=X1,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=y1,
    seq=torch.tensor(all_sequence_outputsnew, dtype=torch.float32),
    pause=torch.tensor(merged_df['High_Pause_Countskd'].values, dtype=torch.float32)
)
print('添加自环前：', test_data)

# 添加自环
test_data.edge_index, test_data.edge_attr = add_self_loops(test_data.edge_index, test_data.edge_weight)
print('\n添加自环后：', test_data)



In [ ]:
class NeuralGraph(nn.Module):
    def __init__(self):
        super(NeuralGraph, self).__init__()
        self.fc_x = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.fc_paired = nn.Sequential(
            nn.Linear(1, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.encoder = nn.Sequential(
            nn.Linear(9216, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.conv = SAGEConv(32, 32, aggr='sum')
        self.conv_activation = nn.Sequential(
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(0.2)
        )

        self.regressor = nn.Linear(32, 1)
        self.regressor_activation = nn.Sequential(
            nn.ReLU()
        )
    def forward(self, data):
        x = data.x
        seq_embedding = data.seq
        pausescore = data.pause
        #print(f"pausescore shape: {pausescore.shape}")

        x = self.fc_x(x) + self.encoder(seq_embedding)
        x = torch.cat((x, self.fc_paired(pausescore.view(-1, 1))), dim=1)
        x = self.fc(x)
        
        # Graph convolution layer
        z = self.conv(x, data.edge_index)
        z = self.conv_activation(z)
        
        # Regressor layer
        out = self.regressor(z)
        out=self.regressor_activation(out)
        return out, z

In [ ]:
# 训练与验证
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
neural_net = NeuralGraph().to(device)
optimizer = optim.Adam(neural_net.parameters(), lr=3e-4)
criterion = nn.MSELoss()
train_data = train_data.to(device)
test_data = test_data.to(device)

In [ ]:

# # 损失函数和优化器
# optimizer = optim.Adam(neural_net.parameters(), lr=0.005)
# criterion = nn.MSELoss()

# num_epochs = 5000

# train_data = train_data.to(device)
# test_data = test_data.to(device)

# for epoch in range(num_epochs):
#     neural_net.train()

#     optimizer.zero_grad()

#     # Unpack predictions (x) and latent representations (z) from the forward pass
#     y_pred, z_train = neural_net(train_data)
#     y_pred = y_pred.view(-1)
    
#     # Compute loss based on predictions
#     loss = criterion(y_pred, train_data.y)
#     loss.backward()
#     optimizer.step()
    
#     if epoch % 1 == 0:
#         neural_net.eval()
#         with torch.no_grad():
#             # Get predictions and latent representations for training data
#             y_train_pred, z_train = neural_net(train_data)
#             y_train_pred = y_train_pred.view(-1).cpu()
#             train_r2 = r2_score(y_train_pred.numpy(), train_data.y.cpu().numpy())
            
#             # Get predictions and latent representations for test data
#             y_test_pred, z_test = neural_net(test_data)
#             y_test_pred = y_test_pred.view(-1).cpu()
#             test_r2 = r2_score(y_test_pred.numpy(), test_data.y.cpu().numpy())
        
#         print(f'Epoch: {epoch}, Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}')


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv  # 使用SAGEConv替代GCNConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import pandas as pd
from src.utils import set_seed
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops

model=NeuralGraph().to(device)
#model.load_state_dict(torch.load('./model/pyg_bertexppausesage0.57.pth', map_location=torch.device('cpu')))
neural_net.load_state_dict(torch.load('./models/bulk_known_seed12_best.pt'))

neural_net = neural_net.to(device)
# 将模型转移到指定设备
#model = model.to(device)

# 将模型切换到评估模式（可选）
model.eval()

In [ ]:
from torch_geometric.utils import negative_sampling

# 正样本：现有的PPI边
positive_edge_index = train_data.edge_index

# 负样本：从未连接的节点对中采样
num_nodes = train_data.num_nodes
negative_edge_index = negative_sampling(
    edge_index=positive_edge_index,
    num_nodes=num_nodes,
    num_neg_samples=positive_edge_index.size(1)  # 生成与正样本数相同的负样本
)


# mini batch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling

# 假设你已经加载了这些数据
# train_data, positive_edge_index, device

# 自定义数据集
class EdgeDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x1, x2 = self.features[idx]
        label = self.labels[idx]
        return x1, x2, label

# 生成节点的嵌入 z
neural_net.eval()
with torch.no_grad():
    _, node_embeddings = neural_net(train_data)  # 只计算一次

# 使用节点嵌入 (z) 来准备正负样本
# 正样本
positive_features = [(node_embeddings[positive_edge_index[0, i]].unsqueeze(0), 
                      node_embeddings[positive_edge_index[1, i]].unsqueeze(0))
                     for i in range(positive_edge_index.size(1))]

# 生成负样本，数量可以与正样本一致或不同，使用全图上的负采样
negative_edge_index = negative_sampling(
    edge_index=positive_edge_index,  # 当前已有的边
    num_nodes=node_embeddings.size(0),  # 节点数量
    num_neg_samples=positive_edge_index.size(1)  # 负样本数量，与正样本数量相同
)

# 负样本
negative_features = [(node_embeddings[negative_edge_index[0, i]].unsqueeze(0), 
                      node_embeddings[negative_edge_index[1, i]].unsqueeze(0))
                     for i in range(negative_edge_index.size(1))]

# 将正样本和负样本组合在一起
features = positive_features + negative_features
labels = torch.cat([torch.ones(len(positive_features), 1), torch.zeros(len(negative_features), 1)], dim=0)

# 初始化 DataLoader
batch_size = 512  # 可以根据实际需要调整
dataset = EdgeDataset(features, labels)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

# 定义 MLP 模型
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(64, 32),  # 假设每个节点特征是32维
            nn.ReLU(),
            nn.Linear(32, 1),   # 输出1维（表示边的存在概率）
            nn.Sigmoid()        # 使用 Sigmoid 将输出限制在 [0, 1] 区间
        )

    def forward(self, x1, x2):
        # 拼接两个节点的特征
        x = torch.cat((x1, x2), dim=-1)  # 拼接后的维度应该是 [batch_size, 64]
        
        # 输入到 MLP 进行边的预测
        output = self.mlp(x)
        
        # 调整输出维度
        return output.squeeze(-1)  # 维度调整为 [batch_size, 1]


# 初始化模型和优化器
mlp_model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling

# 假设你已经加载了这些数据
# train_data, positive_edge_index, device

# 自定义数据集
class EdgeDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x1, x2 = self.features[idx]
        label = self.labels[idx]
        return x1, x2, label

# 生成节点的嵌入 z
neural_net.eval()
with torch.no_grad():
    _, node_embeddings = neural_net(train_data)  # 只计算一次

# 使用节点嵌入 (z) 来准备正负样本
# 正样本
positive_features = [(node_embeddings[positive_edge_index[0, i]].unsqueeze(0), 
                      node_embeddings[positive_edge_index[1, i]].unsqueeze(0))
                     for i in range(positive_edge_index.size(1))]

# 生成负样本，数量可以与正样本一致或不同，使用全图上的负采样
negative_edge_index = negative_sampling(
    edge_index=positive_edge_index,  # 当前已有的边
    num_nodes=node_embeddings.size(0),  # 节点数量
    num_neg_samples=positive_edge_index.size(1)  # 负样本数量，与正样本数量相同
)

# 负样本
negative_features = [(node_embeddings[negative_edge_index[0, i]].unsqueeze(0), 
                      node_embeddings[negative_edge_index[1, i]].unsqueeze(0))
                     for i in range(negative_edge_index.size(1))]

# 将正样本和负样本组合在一起
features = positive_features + negative_features
labels = torch.cat([torch.ones(len(positive_features), 1), torch.zeros(len(negative_features), 1)], dim=0)

# 初始化 DataLoader
batch_size = 512  # 可以根据实际需要调整
dataset = EdgeDataset(features, labels)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

# 定义 MLP 模型
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(64, 32),  # 假设每个节点特征是32维
            nn.ReLU(),
            nn.Linear(32, 1),   # 输出1维（表示边的存在概率）
            nn.Sigmoid()        # 使用 Sigmoid 将输出限制在 [0, 1] 区间
        )

    def forward(self, x1, x2):
        # 拼接两个节点的特征
        x = torch.cat((x1, x2), dim=-1)  # 拼接后的维度应该是 [batch_size, 64]
        
        # 输入到 MLP 进行边的预测
        output = self.mlp(x)
        
        # 调整输出维度
        return output.squeeze(-1)  # 维度调整为 [batch_size, 1]


# 初始化模型和优化器
mlp_model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# 训练模型
num_epochs = 10000  # 设置为合理的 epoch 数
for epoch in range(num_epochs):
    mlp_model.train()
    total_loss = 0

    for x1, x2, label in dataloader:
        x1, x2, label = x1.to(device), x2.to(device), label.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        predictions = mlp_model(x1, x2)
        
        # 计算损失
        loss = criterion(predictions, label)
        
        # 反向传播和优化
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss:.4f}')


In [ ]:
torch.save(mlp_model, './models/ppi_refinement_best.pt')
#torch.save(neural_net, './models/bulk_unknown_example_full.pt')

In [ ]:
mlp_model = MLP().to(device)

# 加载保存的模型权重
model_path = './models/ppi_refinement_best.pt'
mlp_model = torch.load(model_path)

# 切换到评估模式
mlp_model.eval()

In [ ]:
import torch
from itertools import combinations
import csv

# 1. 生成所有可能的节点对
num_nodes = train_data.num_nodes
candidate_edges = list(combinations(range(num_nodes), 2))  # 所有可能的节点对 (i, j)
edge_candidates = torch.tensor(candidate_edges).t().contiguous()  # Shape: [2, num_candidate_edges]

# 2. 获取节点嵌入
neural_net.eval()
with torch.no_grad():
    _, node_embeddings = neural_net(train_data)

# 3. 为候选边生成特征对
x1_candidates = node_embeddings[edge_candidates[0]]  # Shape: [num_candidate_edges, embedding_dim]
x2_candidates = node_embeddings[edge_candidates[1]]  # Shape: [num_candidate_edges, embedding_dim]

# 4. 使用 MLP 预测边概率
mlp_model.eval()
with torch.no_grad():
    edge_probs = mlp_model(x1_candidates, x2_candidates).squeeze(-1)

# 5. 根据预测概率生成新边
threshold = 0.8

# 确保 edge_probs 在 CPU 上，以匹配 edge_candidates
edge_probs_cpu = edge_probs.cpu()
new_edges = edge_candidates[:, edge_probs_cpu >= threshold]
new_edges = new_edges.t().contiguous()  # Shape: [2, num_new_edges]

# 统计新边的数量

print(f"The new PPI contains {new_edges.shape} edges.")

# 6. 导出新边列表
new_edges_list = new_edges.cpu().numpy().tolist()
with open('new_ppi_edgesnew.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['node1', 'node2'])  # Header
    writer.writerows(new_edges_list)

print("New PPI edges exported to 'new_ppi_edgesnew.csv'")


In [ ]:
ppi_matrix

In [ ]:
import torch
import numpy as np
import csv

# 假设以下变量已定义：
# edge_candidates: [2, num_candidate_edges], 所有候选边
# edge_probs: [num_candidate_edges], 每条候选边的预测概率
# num_nodes: 图中节点的总数

# 1. 初始化邻接矩阵
adj_matrix = torch.zeros((num_nodes, num_nodes))

# 2. 填充邻接矩阵
adj_matrix[edge_candidates[0], edge_candidates[1]] = edge_probs.cpu()
adj_matrix[edge_candidates[1], edge_candidates[0]] = edge_probs.cpu()  # 确保矩阵对称

# 3. 导出邻接矩阵到 CSV 文件
adj_matrix_np = adj_matrix.numpy()  # 转为 NumPy 数组
np.savetxt("full_adj_matrix.csv", adj_matrix_np, delimiter=",", fmt="%.6f")

print("Full adjacency matrix exported to 'full_adj_matrix.csv'")


In [ ]:
ppi_matrix=pd.read_csv('./data/9606ppi_matrix.csv')
ppi_matrix.shape

In [ ]:
adj_matrix_np.shape

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# 读取矩阵
ppi_matrix = pd.read_csv('./data/9606ppi_matrix.csv', header=0).values
adj_matrix_np = np.loadtxt("full_adj_matrix.csv", delimiter=",")

# 计算差异矩阵
delta_matrix = adj_matrix_np - ppi_matrix

# 定义颜色映射
colors = ['#8d91c0', '#ffffff', '#ec9f72']  # 蓝色到白色到橙色
cmap = LinearSegmentedColormap.from_list("delta_cmap", colors)

# 绘制差异热图
plt.figure(figsize=(5, 4))
plt.imshow(delta_matrix, cmap=cmap, aspect='auto', interpolation='nearest')
plt.colorbar(label='Difference (Predicted - True)')
plt.title("Delta Heatmap of Predicted and True PPI Matrices")
plt.xlabel("Node Index")
plt.ylabel("Node Index")
plt.tight_layout()

# 显示热图
plt.show()



In [ ]:
new_edges

In [ ]:
adjacency_matrix = torch.zeros((num_nodes, num_nodes), dtype=torch.int32)

# 填充邻接矩阵，设置对应边的位置为1
adjacency_matrix[new_edges[:, 0], new_edges[:, 1]] = 1
adjacency_matrix[new_edges[:, 1], new_edges[:, 0]] = 1  # 无向图需要对称


In [ ]:

ppi_matrix = adjacency_matrix

# 统计非零元素的数量
num_edges = np.count_nonzero(ppi_matrix)

# 如果矩阵是对称的（无向图），每条边会被计算两次（对称位置），所以要除以 2
num_edges = num_edges // 2
num_edges

In [ ]:
#np.savetxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', correlation_matrix, delimiter=',')
coexp=np.loadtxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', delimiter=',')
coexp

In [ ]:
#原始ppi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 读取PPI矩阵

# 获取PPI矩阵中值为1的索引位置
indices = np.where(ppi_matrix == 1)

# 使用这些索引从共表达矩阵中提取相应的值
values = coexp[indices]
filtered_values = values[values != 0]
# 绘制boxplot
plt.boxplot(filtered_values)
plt.title('Boxplot of Co-expression Values at PPI Matrix 1 Locations')
plt.ylabel('Co-expression Value')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载 PPI 矩阵
ppi_matrix = pd.read_csv('./data/9606ppi_matrix.csv', index_col=0)
ppi_matrix_array = ppi_matrix.to_numpy()

# 创建 PPI 矩阵中的边集合
ppi_edges_set = set(zip(*np.where(ppi_matrix_array > 0)))

# 假设 new_ppi_edges 是你的新边列表
# new_ppi_edges 格式为 [(i1, j1), (i2, j2), ...]
new_edges_set = set(map(tuple, new_edges_list))  # new_edges_list 之前保存过

# 找出在原始 PPI 矩阵中不存在但在 new_ppi_edges 中存在的边
new_unique_edges = new_edges_set - ppi_edges_set

# 加载共表达矩阵
coexp_matrix = np.loadtxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', delimiter=',')

# 提取 new_unique_edges 中的共表达值
new_edge_values = []
for edge in new_unique_edges:
    i, j = edge
    value = coexp_matrix[i, j]
    if value != 0:  # 忽略值为 0 的共表达值
        new_edge_values.append(value)

# 转换为 DataFrame 以适应 seaborn
new_edge_values_df = pd.DataFrame(new_edge_values, columns=["Co-expression Value"])

# 设置绘图风格
plt.figure(figsize=(6, 5))
sns.boxplot(
    data=new_edge_values_df, 
    y="Co-expression Value", 
    color="white",  # 设置颜色为白色（透明）
    width=0.3,      # 调整宽度
    linewidth=1.5,  # 设置边框线条粗细
    fliersize=2     # 调整异常值点的大小
)

plt.title("Boxplot of Co-expression Values for New Unique Edges")
plt.ylabel("Co-expression Value")

# 去除 x 轴标签
plt.xticks([])

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

# 加载 PPI 矩阵
ppi_matrix = pd.read_csv('./data/9606ppi_matrix.csv', index_col=0)
ppi_matrix_array = ppi_matrix.to_numpy()

# 创建 PPI 矩阵中的边集合
ppi_edges_set = set(zip(*np.where(ppi_matrix_array > 0)))

# 获取节点数
num_nodes = ppi_matrix_array.shape[0]

# 加载共表达矩阵
coexp_matrix = np.loadtxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', delimiter=',')

# 随机采样10万条边（节点对），确保不在 PPI 边中
random_edges = set()
while len(random_edges) < 100000:
    i = random.randint(0, num_nodes - 1)
    j = random.randint(0, num_nodes - 1)
    # 排除自环和 PPI 中已有的边
    if i != j and (i, j) not in ppi_edges_set and (j, i) not in ppi_edges_set:
        random_edges.add((i, j))

# 提取随机边的共表达值
random_edge_values = []
for edge in random_edges:
    i, j = edge
    value = coexp_matrix[i, j]
    if value != 0:  # 忽略值为 0 的共表达值
        random_edge_values.append(value)

# 转换为 DataFrame 以适应 seaborn
random_edge_values_df = pd.DataFrame(random_edge_values, columns=["Co-expression Value"])

# 设置绘图风格
plt.figure(figsize=(6, 5))
sns.boxplot(
    data=random_edge_values_df, 
    y="Co-expression Value", 
    color="white",  # 设置颜色为白色（透明）
    width=0.2,      # 调整宽度
    linewidth=1.5,  # 设置边框线条粗细
    fliersize=2     # 调整异常值点的大小
)

plt.title("Boxplot of Co-expression Values for Randomly Sampled Non-PPI Edges")
plt.ylabel("Co-expression Value")

# 去除 x 轴标签
plt.xticks([])

plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu
import random

# 加载 PPI 和共表达矩阵
ppi_matrix = pd.read_csv('./data/9606ppi_matrix.csv', index_col=0)
ppi_matrix_array = ppi_matrix.to_numpy()
coexp_matrix = np.loadtxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', delimiter=',')

# 图1：原始 PPI 边的共表达值
indices = np.where(ppi_matrix_array == 1)
values = coexp_matrix[indices]
filtered_values = values[values != 0]

# 图2：新 PPI 边的共表达值
ppi_edges_set = set(zip(*np.where(ppi_matrix_array > 0)))
new_edges_set = set(map(tuple, new_edges_list))  # `new_edges_list` 之前保存过
new_unique_edges = new_edges_set - ppi_edges_set

new_edge_values = []
for edge in new_unique_edges:
    i, j = edge
    value = coexp_matrix[i, j]
    if value != 0:
        new_edge_values.append(value)

# 图3：随机抽取的非 PPI 边的共表达值
random_edges = set()
num_nodes = ppi_matrix_array.shape[0]
while len(random_edges) < 100000:
    i = random.randint(0, num_nodes - 1)
    j = random.randint(0, num_nodes - 1)
    if i != j and (i, j) not in ppi_edges_set and (j, i) not in ppi_edges_set:
        random_edges.add((i, j))

random_edge_values = []
for edge in random_edges:
    i, j = edge
    value = coexp_matrix[i, j]
    if value != 0:
        random_edge_values.append(value)

# 合并三个数据集
data = {
    'Original PPI': filtered_values,
    'New Unique PPI': new_edge_values,
    'Random Non-PPI': random_edge_values
}

# 将数据转换为 DataFrame
all_data = []
labels = []

for key in data:
    all_data.extend(data[key])
    labels.extend([key] * len(data[key]))

df = pd.DataFrame({'Co-expression Value': all_data, 'Group': labels})

# 绘制合并的箱线图
plt.figure(figsize=(10, 7))
sns.boxplot(x='Group', y='Co-expression Value', data=df, width=0.5)

# 计算 p 值并显示星号
p1 = mannwhitneyu(data['Original PPI'], data['New Unique PPI']).pvalue
p2 = mannwhitneyu(data['Original PPI'], data['Random Non-PPI']).pvalue
p3 = mannwhitneyu(data['New Unique PPI'], data['Random Non-PPI']).pvalue

# 在图中添加 p 值的星号
def add_stat_annotation(ax, x1, x2, y, h, text):
    """Helper function to draw p-value annotations on boxplots."""
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, color='black')
    ax.text((x1 + x2) * .5, y + h, text, ha='center', va='bottom', color='black')

ax = plt.gca()
y_max = df['Co-expression Value'].max()
h = y_max * 0.05

# Add annotations for p-values
add_stat_annotation(ax, 0, 1, y_max, h, "*" if p1 < 0.05 else "ns")
add_stat_annotation(ax, 0, 2, y_max + h * 1.5, h, "*" if p2 < 0.05 else "ns")
add_stat_annotation(ax, 1, 2, y_max + h * 3, h, "*" if p3 < 0.05 else "ns")

plt.title('Boxplot of Co-expression Values for Different Edge Types')
plt.ylabel('Co-expression Value')
plt.show()


In [ ]:
p3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载 PPI 矩阵
ppi_matrix = pd.read_csv('./data/9606ppi_matrix.csv', index_col=0)
ppi_matrix_array = ppi_matrix.to_numpy()

# 创建 PPI 矩阵中的边集合
ppi_edges_set = set(zip(*np.where(ppi_matrix_array > 0)))

# 假设 new_ppi_edges 是你的新边列表
# new_ppi_edges 格式为 [(i1, j1), (i2, j2), ...]
new_edges_set = set(map(tuple, new_edges_list))  # new_edges_list 之前保存过

# 找出在原始 PPI 矩阵中不存在但在 new_ppi_edges 中存在的边
new_unique_edges = new_edges_set - ppi_edges_set

# 加载共表达矩阵
coexp_matrix = np.loadtxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', delimiter=',')

# 提取 new_unique_edges 中的共表达值
new_edge_values = []
for edge in new_unique_edges:
    i, j = edge
    value = coexp_matrix[i, j]
    if value != 0:  # 忽略值为 0 的共表达值
        new_edge_values.append(value)

# 转换为 DataFrame 以适应 seaborn
new_edge_values_df = pd.DataFrame(new_edge_values, columns=["Co-expression Value"])

# 绘制箱线图
plt.figure(figsize=(6, 5))
sns.boxplot(data=new_edge_values_df, y="Co-expression Value")
plt.title("Boxplot of Co-expression Values for New Unique Edges")
plt.ylabel("Co-expression Value")
plt.show()


In [ ]:
# torch.save(mlp_model, './model/240927mlp_model70epochloss0.45.pth')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
values = edge_probs_cpu.flatten()
filtered_values = values[values != 0]
plt.boxplot(filtered_values) 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 加载 PPI 矩阵
ppi_matrix = pd.read_csv('./data/9606ppi_matrix.csv', index_col=0)
ppi_matrix_array = ppi_matrix.to_numpy()

# 创建 PPI 矩阵中的边集合
ppi_edges_set = set(zip(*np.where(ppi_matrix_array > 0)))

# 假设 new_ppi_edges 是你的新边列表
# new_ppi_edges 格式为 [(i1, j1), (i2, j2), ...]
new_edges_set = set(map(tuple, new_edges_list))  # new_edges_list 之前保存过

# 找出在原始 PPI 矩阵中不存在但在 new_ppi_edges 中存在的边
new_unique_edges = new_edges_set - ppi_edges_set

# 加载共表达矩阵
coexp_matrix = np.loadtxt('<RECIPE_PROJECT_ROOT>/data/240917co_expression_network_matrix.csv', delimiter=',')

# 提取 new_unique_edges 中的共表达值
new_edge_values = []
for edge in new_unique_edges:
    i, j = edge
    value = coexp_matrix[i, j]
    if value != 0:  # 忽略值为 0 的共表达值
        new_edge_values.append(value)

# 转换为 DataFrame 以适应 seaborn
new_edge_values_df = pd.DataFrame(new_edge_values, columns=["Co-expression Value"])

# 绘制箱线图
plt.figure(figsize=(6, 5))
sns.boxplot(data=new_edge_values_df, y="Co-expression Value")
plt.title("Boxplot of Co-expression Values for New Unique Edges")
plt.ylabel("Co-expression Value")
plt.show()


In [ ]:
import numpy as np
ppi_df = pd.read_csv("./data/9606ppi_matrix.csv", index_col=0)
# 将 DataFrame 转换为 NumPy 数组
ppi_matrix = ppi_df.to_numpy()

# 统计非零元素的数量
num_edges = np.count_nonzero(ppi_matrix)

# 如果矩阵是对称的（无向图），每条边会被计算两次（对称位置），所以要除以 2
num_edges = num_edges // 2

print(f"The PPI matrix contains {num_edges} edges.")


# 统计个数

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
absolute_correlation_matrix=coexp
# 假设 absolute_correlation_matrix 是你的绝对相关性矩阵
# 设置阈值
threshold = 0.97

# 应用阈值，将小于阈值的元素设为 0
binary_matrix = np.where(np.abs(absolute_correlation_matrix) >= threshold, absolute_correlation_matrix, 0)

# 将处理后的矩阵转换为稀疏矩阵
absolute_correlation_matrix_sparse = csr_matrix(binary_matrix)


# 假设 absolute_correlation_matrix_sparse 是你的稀疏矩阵
# 计算非零元素的数量
num_edges = absolute_correlation_matrix_sparse.nnz

print(f"边的数量: {num_edges}")
# 打印结果以确认
print("稀疏矩阵形状:", absolute_correlation_matrix_sparse.shape)
print("稀疏矩阵示例:", absolute_correlation_matrix_sparse)

# 如果需要将稀疏矩阵保存到文件中
# from scipy.io import savemat
# savemat('absolute_correlation_matrix_sparse.mat', {'sparse_matrix': absolute_correlation_matrix_sparse})


In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# 获取稀疏矩阵的非零元素的索引 (即边的坐标)
sparse_edges = set(zip(*absolute_correlation_matrix_sparse.nonzero()))

# 找到 PPI 网络 new_unique_edges 和 coexp_matrix 中存在的边的交集
common_edges = new_unique_edges.intersection(sparse_edges)

# 输出交集的边数
print(f"共同边的数量: {len(common_edges)}")

# 打印共同边的例子
print("共同边的示例:", list(common_edges)[:10])

# 如果需要从 absolute_correlation_matrix_sparse 中提取这些共同边的值：
common_edge_values = [absolute_correlation_matrix_sparse[i, j] for (i, j) in common_edges]

# 打印一些共同边的共表达值
print("共同边的共表达值示例:", common_edge_values[:10])


In [ ]:
#torch.save(mlp_model, './model/240926mlp_model70epochloss0.45.pth')


In [ ]:
# 初始化 MLP 模型、损失函数和优化器
mlp_model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.005)
num_epochs = 5000
# 开始训练
for epoch in range(num_epochs):
    # 在每个 epoch 开始时重新生成节点嵌入
    _, node_embeddings = neural_net(train_data)
    
    # 使用节点嵌入 (z) 来准备正负样本
    positive_features = [(node_embeddings[positive_edge_index[0, i]].unsqueeze(0), 
                          node_embeddings[positive_edge_index[1, i]].unsqueeze(0))
                         for i in range(positive_edge_index.size(1))]

    negative_features = [(node_embeddings[negative_edge_index[0, i]].unsqueeze(0), 
                          node_embeddings[negative_edge_index[1, i]].unsqueeze(0))
                         for i in range(negative_edge_index.size(1))]

    # 将正样本和负样本组合在一起
    features = positive_features + negative_features
    labels = torch.cat([torch.ones(len(positive_features), 1), torch.zeros(len(negative_features), 1)], dim=0)

    # 重新初始化 DataLoader
    dataset = EdgeDataset(features, labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    mlp_model.train()
    total_loss = 0

    # 迭代每个 minibatch
    for batch in dataloader:
        optimizer.zero_grad()  # 清除累积梯度
        
        # 获取 batch 数据
        x1_batch, x2_batch, label_batch = batch
        
        # 移动到设备
        x1_batch = x1_batch.to(device)
        x2_batch = x2_batch.to(device)
        label_batch = label_batch.to(device)

        # 对 batch 中的所有 (x1, x2) 进行前向传播
        predictions = mlp_model(x1_batch, x2_batch)

        # 调整尺寸以匹配 label_batch
        predictions = predictions.squeeze(-1)

        # 计算损失
        loss = criterion(predictions, label_batch)
        
        # 反向传播
        loss.backward()  # 不需要设置 retain_graph=True
        optimizer.step()
        
        # 累积 loss
        total_loss += loss.item()

    # 打印每个 epoch 的平均损失
    if epoch % 10 == 0:  # 每隔 10 个 epoch 打印一次损失
        avg_loss = total_loss / len(dataloader)
        print(f'Epoch {epoch}, Loss: {avg_loss:.4f}')



# 正常训练

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(64, 32),  # 假设每个节点特征是32维
            nn.ReLU(),
            nn.Linear(32, 1),   # 输出1维（表示边的存在概率）
            nn.Sigmoid()        # 使用Sigmoid将输出限制在[0, 1]区间
        )

    def forward(self, x1, x2):
        # 如果 x1 或 x2 是一维的，将它们转换为二维
        if x1.dim() == 1:
            x1 = x1.unsqueeze(0)
        if x2.dim() == 1:
            x2 = x2.unsqueeze(0)
            
        # 拼接两个节点的特征
        x = torch.cat((x1, x2), dim=-1)  # 拼接后的维度应该是 [1, 64]
        
        # 输入到MLP进行边的预测
        return self.mlp(x)


In [ ]:
# 生成节点的嵌入 z
_, node_embeddings = neural_net(train_data)  # node_embeddings 是节点的嵌入 z

# 使用节点嵌入 (z) 来准备正负样本
positive_features = [(node_embeddings[positive_edge_index[0, i]].unsqueeze(0), 
                      node_embeddings[positive_edge_index[1, i]].unsqueeze(0))
                     for i in range(positive_edge_index.size(1))]

negative_features = [(node_embeddings[negative_edge_index[0, i]].unsqueeze(0), 
                      node_embeddings[negative_edge_index[1, i]].unsqueeze(0))
                     for i in range(negative_edge_index.size(1))]

# 定义正负样本的标签
positive_labels = torch.ones(len(positive_features), 1)
negative_labels = torch.zeros(len(negative_features), 1)

# 将正样本和负样本组合在一起
features = positive_features + negative_features
labels = torch.cat([positive_labels, negative_labels], dim=0)

# 定义模型、损失函数和优化器
mlp_model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.005)
num_epochs = 5000

# 开始训练
for epoch in range(num_epochs):
    mlp_model.train()
    
    optimizer.zero_grad()

    # 计算所有正负样本的预测
    predictions = []
    for x1, x2 in features:
        pred = mlp_model(x1.to(device), x2.to(device))
        predictions.append(pred)
    
    predictions = torch.cat(predictions, dim=0)
    
    # 计算损失
    loss = criterion(predictions, labels.to(device))
    
    # 反向传播
    loss.backward()  # 确保只调用一次 .backward()
    optimizer.step()

    if epoch % 1 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')


In [ ]:
# 定义模型、损失函数和优化器
mlp_model = MLP().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.005)
num_epochs = 5000
# 开始训练
for epoch in range(num_epochs):
    mlp_model.train()
    
    optimizer.zero_grad()

    # 计算所有正负样本的预测
    predictions = []
    for x1, x2 in features:
        pred = mlp_model(x1.to(device), x2.to(device))
        predictions.append(pred)
    
    predictions = torch.cat(predictions, dim=0)
    
    # 计算损失
    loss = criterion(predictions, labels.to(device))
    loss.backward()
    optimizer.step()

    if epoch % 1 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')


In [ ]:
from torch_geometric.data import DataLoader
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

model_instance= NeuralNet().to(device)
# 创建DataLoader实例
train_loader = DataLoader([train_data], batch_size=1, shuffle=False)

def extract_embeddings(model, loader):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)  # 确保这里data是单个Data对象
            embedding = model.encoder(data.seq)  # 提取encoder部分的输出作为嵌入
            embeddings.append(embedding.cpu().numpy())
    return np.vstack(embeddings)

# 使用DataLoader提取嵌入
embeddings = extract_embeddings(model_instance, train_loader)  # 确保传递的是DataLoader实例

# 计算节点间的相似性
from sklearn.metrics.pairwise import cosine_similarity
similarity_matrix = cosine_similarity(embeddings)

# 根据相似性阈值确定新的边
threshold = 0.92  # 设置一个阈值
new_edges = np.where(similarity_matrix > threshold)

# 创建新的PPI网络，此处简单地输出边的列表
new_ppi_edges = list(zip(new_edges[0], new_edges[1]))#

